In [1]:
# 1. Instalacja wymaganego narzędzia do dekompresji
!apt-get update && apt-get install -y zstd

# 2. Instalacja Ollama i niezbędnych bibliotek, w tym Gradio (do GUI)
!curl -fsSL https://ollama.com/install.sh | sh
!pip install ollama pymupdf sentence-transformers faiss-cpu gradio

import subprocess
import time

# 3. Uruchomienie lokalnego serwera Ollama w tle
print("\nUruchamianie lokalnego serwera Ollama...")
subprocess.Popen(["ollama", "serve"])
time.sleep(5) # Czekamy 5 sekund na pełny start serwera

# 4. Pobranie modelu
print("Pobieranie modelu llama3.2 (proszę czekać)...")
!ollama pull llama3.2
print("Gotowe! Silnik LLM jest przygotowany.")

Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:3 https://cli.github.com/packages stable/main amd64 Packages [354 B]
Get:4 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:6 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:7 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:8 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:9 http://security.ubuntu.com/ubuntu jammy-security/restricted amd64 Packages [7,221 kB]
Get:10 http://security.ubuntu.com/ubuntu jammy-security/universe amd64 Packages [1,307 kB]
Get:11 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [4,026 kB]
Get:12 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Packages [3,066 kB]
Hit:13 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu j

In [2]:
import os
import faiss
import numpy as np
from tqdm import tqdm
import ollama
import pymupdf
from sentence_transformers import SentenceTransformer
import warnings
warnings.filterwarnings("ignore")

print("Ładowanie modelu wektoryzującego...")
# Model automatycznie użyje karty GPU, jeśli została włączona
embedder = SentenceTransformer('paraphrase-multilingual-mpnet-base-v2')
index = faiss.IndexFlatL2(embedder.get_embedding_dimension())
metadata = []

class AdminRAG:
    def __init__(self, embedder, index, metadata, chunk_size=400):
        self.embedder = embedder
        self.index = index
        self.metadata = metadata
        self.chunk_size = chunk_size

    def process_file(self, file_path):
        text_data = []

        # Obsługa plików TXT
        if file_path.endswith('.txt'):
            with open(file_path, 'r', encoding='utf-8') as f:
                text_data.append((0, f.read().replace('\n', ' ')))

        # Obsługa plików PDF
        elif file_path.endswith('.pdf'):
            pdf_document = pymupdf.open(file_path)
            for page_num in range(len(pdf_document)):
                page = pdf_document.load_page(page_num)
                text_data.append((page_num, str(page.get_text()).replace("\n", " ")))
        else:
            return

        chunks = []
        for page_num, page_text in text_data:
            page_chunks = [(page_num, page_text[i:i+self.chunk_size]) for i in range(0, len(page_text), self.chunk_size)]
            chunks.extend(page_chunks)

        for chunk_num, (page_number, chunk) in enumerate(tqdm(chunks, desc=f"Dodawanie: {os.path.basename(file_path)}")):
            embeddings = self.embedder.encode(chunk, show_progress_bar=False)
            self.index.add(np.array([embeddings]))
            self.metadata.append({
                "filename": os.path.basename(file_path),
                "page_number": page_number,
                "chunk": chunk
            })

    def answer_question(self, query):
        q_embedding = self.embedder.encode(query, show_progress_bar=False)
        D, I = self.index.search(np.array([q_embedding]), 4)
        chunks = [self.metadata[i] for i in I[0]]

        context = ""
        for i, chunk in enumerate(chunks):
            context += f"Dokument [{i+1}]: {chunk['chunk']}\n"

        # Restrykcyjny prompt systemowy dla Administratora IT
        messages = [
            {
                "role": "system",
                "content": (
                    "Jesteś zaawansowanym asystentem wspierającym administrację systemami operacyjnymi i sieciami komputerowymi. "
                    "Twoim zadaniem jest pomaganie inżynierom poprzez precyzyjne wskazywanie komend i konfiguracji. "
                    "Odpowiadaj w języku polskim TYLKO i wyłącznie na podstawie dostarczonej dokumentacji z kontekstu. "
                    "Jeśli w kontekście brakuje rozwiązania, napisz DOKŁADNIE: "
                    "'Brak informacji w wczytanej dokumentacji systemowej.' "
                    "Kategorycznie zabraniam zmyślania komend lub wymyślania konfiguracji, których nie ma w plikach."
                )
            },
            {"role": "user", "content": f"Wczytana dokumentacja:\n{context}\n\nProblem zgłoszony przez inżyniera: {query}"}
        ]

        response = ollama.chat(model="llama3.2", messages=messages, options={"temperature": 0.1})
        return response['message']['content'], chunks

rag = AdminRAG(embedder, index, metadata)

Ładowanie modelu wektoryzującego...


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/5.12k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/402 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
import os
import gradio as gr

# Inteligentne szukanie folderu knowledge
knowledge_dir = "knowledge/"
if not os.path.exists(knowledge_dir):
    if os.path.exists("sample_data/knowledge/"):
        knowledge_dir = "sample_data/knowledge/"
    else:
        os.makedirs(knowledge_dir)
        print(f"Utworzono pusty folder '{knowledge_dir}'. Wrzuć tam pliki!")

# Wczytywanie plików z folderu do bazy wektorowej FAISS
if os.path.exists(knowledge_dir) and len(os.listdir(knowledge_dir)) > 0:
    print(f"Inicjalizacja bazy wiedzy z folderu: {knowledge_dir} ...")
    for file in os.listdir(knowledge_dir):
        rag.process_file(os.path.join(knowledge_dir, file))
    print(f"\nGotowe! Liczba wektorów w bazie FAISS: {index.ntotal}")
else:
    print("UWAGA: Folder z bazą wiedzy jest pusty. Wrzuć pliki txt/pdf przed zadawaniem pytań.")

# Funkcja łącząca nasz kod z interfejsem graficznym
def chat_with_rag(query):
    odpowiedz, zrodla = rag.answer_question(query)

    zrodla_text = ""
    for i, chunk in enumerate(zrodla):
        fragment = chunk['chunk'][:250].strip() + "..."
        zrodla_text += f"[{i+1}] Plik: {chunk['filename']}\n{fragment}\n\n"

    return odpowiedz, zrodla_text

# Definicja aplikacji Gradio
demo = gr.Interface(
    fn=chat_with_rag,
    inputs=gr.Textbox(
        lines=3,
        placeholder="Opisz problem inżynierski lub wpisz pytanie (np. 'Jak skonfigurować routing OSPF?')...",
        label="Zapytanie do bazy wiedzy"
    ),
    outputs=[
        gr.Textbox(label="Odpowiedź Admin-RAG", lines=10),
        gr.Textbox(label="Znalezione w dokumentacji (Źródła)", lines=6)
    ],
    title="🛠️ Admin-RAG: Asystent Infrastruktury IT",
    description="Wpisz zapytanie, aby przeszukać lokalną dokumentację systemową, sieciową (Cisco) i konteneryzacyjną.",
    theme="soft",
    allow_flagging="never"
)

# Start aplikacji - wygeneruje publiczny link webowy!
demo.launch(share=True, debug=True)

Inicjalizacja bazy wiedzy z folderu: knowledge/ ...


Dodawanie: kubernetes_k8s_administracja.txt: 100%|██████████| 4/4 [00:00<00:00, 68.99it/s]



Gotowe! Liczba wektorów w bazie FAISS: 31
Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://95b5da98222e7c8122.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
